# 人工审阅流程：批量流程输出 → 审阅包 → 笔记本上逐帧看一遍并修正 ID / 头尾 → 应用修正 → 和机器对比

批量流程（`batch_pipeline.ipynb`）做完 3D 之后，在这里准备“审阅包”，拿到任何一台电脑（包括笔记本，不需要显卡）上用审阅界面把每一帧都看一遍，修正身份（F / M1 / M2）和头尾。

| 步骤 | 做什么 | 在哪里跑 | 用时 |
|---|---|---|---|
| 1–2 | 参数、定位路径 | notebook | 秒级 |
| 3 | 状态总表：代理视频、3D、审阅数据、审阅进度 | notebook | 随时重跑 |
| 4 | 启动后台审阅包生成（一直运行，最低优先级；本机推理运行时代理视频自动暂停） | 本机 GPU 编码器 | 代理视频 ~35 分钟/实验；审阅数据几分钟/实验 |
| 5 | 前台给单个实验生成审阅数据（例如 20 秒小测试） | 本机 CPU | 秒～分钟 |
| 6 | 看机器标出的风险点和头尾自动修正摘要 | notebook | 秒级 |
| 7 | 复制审阅包到移动硬盘 / 把修正记录拷回 | notebook | 每实验 ~3 GB |
| 8 | 在本机打开审阅界面（笔记本上的用法见最后一节） | 本机 | — |
| 9 | 应用修正 → `<标签>_3d_reviewed.csv` | notebook | 秒级 |
| 10 | 机器风险点 vs 人工修正 对比 | notebook | 秒级 |
| 11 | 停止后台任务 | notebook | 秒级 |

**背后的脚本**（都在 `scripts/review/`，说明见 `scripts/review/README.md`）：
- `proxies.py` — 三个视角的代理视频：800×800；`*.play.mp4` 每 6 帧取 1 帧（166 fps → 27.7 fps，第 k 帧 = 原始第 6k 帧），`*.full.mp4` 每帧都有（逐帧检查用）
- `review_data.py` — 取回 3D 用到的嘴 / 胸鳍 / 尾 2D 点，自动修正头尾，逐节点三角化，生成机器风险点 `machine_risk.csv`
- `build_packages.py` — 后台一直运行：先做已完成 3D 的审阅数据，再一个一个做代理视频
- `review_gui.py` — 审阅界面（只需要 numpy / pandas / opencv / PyQt5）
- `ledger.py` — 修正记录、播放覆盖记录、应用修正、对比

**修正记录**（审阅界面自动写，按键即保存）：
- `human_events.csv`：每次修正（身份 / 头尾 / 不确定 / 3D 不可信），含按键帧和自动对齐后的帧、所在风险点
- `coverage.csv`：哪些帧以什么速度播放过（≤1 倍速才算“看过”）
- `session_log.csv`：每次审阅的开始、结束、时长
- `machine_risk.csv`：机器风险点，生成后不再改，用来和人工修正对比

**注意**：
- 换了 3D 结果（新的 `ROUND_TAG`）就是新的审阅文件夹；旧标签的修正不会套到新结果上
- 同一个实验不要同时在两台电脑上审（会弹出提示）
- 修正记录是花大量人工时间换来的唯一成果：审完一段就用步骤 7 拷回 Z:

## 步骤 1 — 参数

只需要改这一格。
- `EXPERIMENTS`：要准备审阅包的实验（完整文件夹名或 `P12`、`ALL` 这样的组名；顺序 = 优先顺序）
- `ROUND_TAG`：3D 结果的标签（和 `batch_pipeline.ipynb` 一致）；`SAMPLE_TAG` 是 20 秒小测试的标签，用来先试界面
- `REVIEW_ROOT`：审阅包放在哪里（`None` = `Z:\medaka\review_packages`）
- `SSD_ROOT`：移动硬盘上的审阅文件夹，例如 `"E:/medaka_review"`（`None` = 不复制）
- `RUN_BUILDER` / `OPEN_GUI` 为 `True` 时步骤 4 / 8 才真正启动（防止“全部运行”误启动）

In [1]:
EXPERIMENTS = ["20260112_P_1_1_FCMC"]   # 先只做一个实验；确认没问题后改成 ["P12", "ALL"]
ROUND_TAG   = "s14_bottomup_3node_reviewed"
SAMPLE_TAG  = "s14_bottomup_3node_reviewed_test"   # 20 秒小测试（中间 20 秒），先用它试界面

REVIEW_ROOT  = None          # None = <data_root>/review_packages
SSD_ROOT     = None          # 例如 "E:/medaka_review"
PATHS_CONFIG = None          # None = 自动选择

RUN_BUILDER = False          # 步骤 4：后台生成审阅包
OPEN_GUI    = False          # 步骤 8：在本机打开审阅界面
GUI_EXPERIMENT = "20260112_P_1_1_FCMC"
GUI_TAG        = SAMPLE_TAG

## 步骤 2 — 定位仓库根目录和本机路径配置

- 输入：当前目录、`configs/paths.<用户名>.toml` 或 `configs/paths.local.toml`
- 输出：`REPO`、`CFG`、`PATHS`、`REVIEW_ROOT`、`LOG_DIR`

In [2]:
import os
import sys
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for d in [start, *start.parents]:
        if (d / "pyproject.toml").exists() and (d / "scripts" / "common" / "local_paths.py").exists():
            return d
    raise FileNotFoundError(f"找不到仓库根目录（从 {start} 向上没有 pyproject.toml）")


REPO = find_repo_root(Path.cwd().resolve())
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.common.local_paths import load_local_paths
from scripts.sleap.batch_bottomup_track import default_paths_config

CFG = Path(PATHS_CONFIG) if PATHS_CONFIG else default_paths_config()
PATHS = load_local_paths(CFG)
if PATHS is None:
    raise FileNotFoundError(f"路径配置不存在：{CFG}")
REVIEW_ROOT = Path(REVIEW_ROOT) if REVIEW_ROOT else PATHS.data_root / "review_packages"
LOG_DIR = PATHS.runs_dir / "review_packages"
for name, value in [("medaka_data_dir", PATHS.medaka_data_dir), ("reconstruction_output_dir", PATHS.reconstruction_output_dir),
                    ("REVIEW_ROOT", REVIEW_ROOT)]:
    print(f"{name:26s} {'✅' if value.exists() else '❌ 不存在'} {value}")
print("仓库根目录 :", REPO)
print("路径配置   :", CFG)
print("Python     :", sys.executable)

medaka_data_dir            ✅ \\bucket.oist.jp\bucket\StephensU\medaka\medaka_202512
reconstruction_output_dir  ✅ \\bucket.oist.jp\bucket\StephensU\medaka\reconstruction_3d
REVIEW_ROOT                ✅ \\bucket.oist.jp\bucket\StephensU\medaka\review_packages
仓库根目录 : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline
路径配置   : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline\configs\paths.ReiterU.toml
Python     : C:\Users\ReiterU\Documents\GitHub\medaka-mating-pipeline\.venv\Scripts\python.exe


## 步骤 3 — 状态总表

随时重新运行。每个实验一行：
- 代理视频：✅ 三个视角都完成；⏳ 正在生成（有 `.partial.mp4`）；空白 = 未开始
- 3D：✅ 批量流程的 3D 已完成（`batch_3d.done`）
- 审阅数据：✅ 已生成；🔄 3D 比审阅数据新，需要重新生成；空白 = 还不能做
- 已看过 / 修正行数：`coverage.csv` 里 ≤1 倍速播放过的比例（相对整段视频），`human_events.csv` 的行数

In [3]:
import json

import pandas as pd

from scripts.common.experiment_sets import resolve_experiments
from scripts.review.ledger import COVERAGE_COLUMNS, EVENT_COLUMNS, covered_fraction, read_csv, watched_intervals

EXPS = resolve_experiments(EXPERIMENTS, PATHS.medaka_data_dir)


def proxy_status(exp):
    d = REVIEW_ROOT / exp / "proxy"
    if (d / "proxy_manifest.json").exists():
        return "✅"
    return "⏳" if any(d.glob("*.partial.mp4")) else ""


def review_status(exp, tag):
    done3d = PATHS.reconstruction_output_dir / exp / tag / "batch_3d.done"
    manifest = REVIEW_ROOT / exp / tag / "review_manifest.json"
    if not manifest.exists():
        return ""
    return "🔄" if done3d.exists() and done3d.stat().st_mtime > manifest.stat().st_mtime else "✅"


def progress(exp, tag):
    folder = REVIEW_ROOT / exp / tag
    pm = REVIEW_ROOT / exp / "proxy" / "proxy_manifest.json"
    if not folder.exists():
        return "", ""
    cov = read_csv(folder / "coverage.csv", COVERAGE_COLUMNS)
    n_ev = len(read_csv(folder / "human_events.csv", EVENT_COLUMNS))
    if not pm.exists():
        return "", n_ev
    n = min(v["source_frames"] for v in json.loads(pm.read_text())["views"].values())
    return f"{100 * covered_fraction(watched_intervals(cov), 0, n - 1):.1f}%", n_ev


rows = []
for exp in EXPS:
    row = {"实验": exp, "代理视频": proxy_status(exp)}
    for tag in (ROUND_TAG, SAMPLE_TAG):
        short = "测试" if tag == SAMPLE_TAG else "正式"
        row[f"3D({short})"] = "✅" if (PATHS.reconstruction_output_dir / exp / tag / "batch_3d.done").exists() else ""
        row[f"审阅数据({short})"] = review_status(exp, tag)
        row[f"已看过({short})"], row[f"修正行数({short})"] = progress(exp, tag)
    rows.append(row)
pd.DataFrame(rows)

,实验,代理视频,3D(正式),审阅数据(正式),已看过(正式),修正行数(正式),3D(测试),审阅数据(测试),已看过(测试),修正行数(测试)
0,20260112_P_1_1_FCMC,✅,,,,,✅,✅,0.0%,0


## 步骤 4 — 启动后台审阅包生成（一直运行）

- 对 `EXPERIMENTS` 循环：先给 3D 已完成的实验生成审阅数据（几分钟），再生成下一个实验的代理视频（~40 分钟），每 10 分钟检查一次新的 3D 结果
- **最低优先级**、一次只转一个视角；**本机 SLEAP 推理运行时自动暂停代理视频**（实测全长转码会让本机推理从 32.5 降到约 29 帧/秒，慢约 11%），推理一结束自动继续；审阅数据（CPU，几分钟）不暂停
- 想不管推理也一直转：在命令里加 `--when-inference run`
- 关掉 notebook 也不会停（重启 / 睡眠会停，重新运行本格即可接着做；做到一半的文件自动作废重做）
- 日志：`runs_dir/review_packages/review_builder.log`
- 本格也定义了步骤 11 用的 `task_alive` / `stop_background`

In [4]:
import subprocess
import time

import psutil

LOG_DIR.mkdir(parents=True, exist_ok=True)
DETACHED = subprocess.CREATE_NEW_PROCESS_GROUP | subprocess.DETACHED_PROCESS if os.name == "nt" else 0


def task_alive(name):
    # 返回仍在运行的后台进程；pid 文件里的启动时间要对得上，避免 PID 被系统复用后误判
    pid_file = LOG_DIR / f"{name}.pid.json"
    if not pid_file.exists():
        return None
    info = json.loads(pid_file.read_text())
    try:
        proc = psutil.Process(info["pid"])
        return proc if abs(proc.create_time() - info["started"]) < 60 else None
    except psutil.Error:
        return None


def start_background(name, cmd):
    if proc := task_alive(name):
        print(f"{name} 已在运行，PID {proc.pid}")
        return
    log = open(LOG_DIR / f"{name}.log", "a", encoding="utf-8")
    log.write(f"\n# started {time.ctime()}\n# cmd: {' '.join(cmd)}\n")
    log.flush()
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, cwd=REPO,
                            creationflags=DETACHED, start_new_session=os.name != "nt")
    (LOG_DIR / f"{name}.pid.json").write_text(json.dumps({"pid": proc.pid, "started": psutil.Process(proc.pid).create_time()}))
    print(f"{name} 已启动，PID {proc.pid}，日志 {LOG_DIR / (name + '.log')}")


def stop_background(name):
    # 连同子进程（ffmpeg）一起结束
    proc = task_alive(name)
    if proc is None:
        print(f"{name} 没有在运行")
        return
    family = [proc, *proc.children(recursive=True)]
    for p in family:
        p.terminate()
    _, alive = psutil.wait_procs(family, timeout=10)
    for p in alive:
        p.kill()
    print(f"{name} 已停止（PID {proc.pid}，共 {len(family)} 个进程）")


if not RUN_BUILDER:
    print("未启动：确认后把步骤 1 的 RUN_BUILDER 改成 True，再运行本格")
    if proc := task_alive("review_builder"):
        print(f"（review_builder 正在运行，PID {proc.pid}）")
else:
    start_background("review_builder", [
        sys.executable, "-u", "-m", "scripts.review.build_packages", *EXPERIMENTS,
        "--paths-config", str(CFG), "--round-tag", ROUND_TAG, "--review-root", str(REVIEW_ROOT), "--watch", "10",
    ])
log = LOG_DIR / "review_builder.log"
if log.exists():
    print("\n".join(log.read_text(encoding="utf-8", errors="replace").splitlines()[-6:]))

未启动：确认后把步骤 1 的 RUN_BUILDER 改成 True，再运行本格


## 步骤 5 — 前台给单个实验生成审阅数据

用于小测试标签（`SAMPLE_TAG`，后台任务不会给它做）或想马上重做某个实验时。已经生成过且 3D 没变的会跳过（`FORCE = True` 强制重做）。
- 输入：`.slp`（三视角）、`<标签>_best.json`、`<标签>_3d.csv`、`<标签>_3d_identified.csv`、标定
- 输出：`REVIEW_ROOT/<实验>/<标签>/review_data.npz`、`machine_risk.csv`、`review_manifest.json`
- 全长实验估计几分钟（读 `.slp` 占大头）；20 秒小测试约 5 秒

In [5]:
from scripts.review.build_packages import review_data_pending, source_fps
from scripts.review.review_data import build_review_data, calibration_folder

BUILD_EXPERIMENT, BUILD_TAG, FORCE = GUI_EXPERIMENT, SAMPLE_TAG, False

out_dir = REVIEW_ROOT / BUILD_EXPERIMENT / BUILD_TAG
recon_dir = PATHS.reconstruction_output_dir / BUILD_EXPERIMENT / BUILD_TAG
if not (recon_dir / "batch_3d.done").exists():
    print(f"3D 还没完成：{recon_dir}")
elif not FORCE and not review_data_pending(BUILD_EXPERIMENT, PATHS, REVIEW_ROOT, BUILD_TAG):
    print(f"已是最新，跳过：{out_dir}")
else:
    build_review_data(BUILD_EXPERIMENT, BUILD_TAG, PATHS.sleap_output_dir / BUILD_EXPERIMENT / BUILD_TAG, recon_dir, out_dir,
                      calibration_folder(recon_dir, BUILD_TAG, PATHS.calibration_model_dir),
                      fps=source_fps(PATHS, BUILD_EXPERIMENT))

已是最新，跳过：\\bucket.oist.jp\bucket\StephensU\medaka\review_packages\20260112_P_1_1_FCMC\s14_bottomup_3node_reviewed_test


## 步骤 6 — 机器风险点和头尾自动修正摘要

- `orientation_frames`：能判断头尾的 帧×鱼 数；`frames_with_any_view_auto_flipped`：至少一个视角的嘴/尾被自动对调过的 帧×鱼 数；`orientation_override_frames`：按时间连续性推翻了多数视角 SLEAP 判断的 帧×鱼 数
- 身长和“游动时头朝前”比例用来检查头尾修正是否合理（鱼体长 25–35 mm）
- 风险点类型：`close` 近距离接近、`jump` 跳变、`gap` 插值断档、`missing_view` 少于 3 个视角、`residual` 三角化误差大、`unbound` 没绑定到 top 轨迹、`orientation` 头尾判断推翻多数视角、`fish_count` 鱼数变化

In [6]:
import warnings

import numpy as np

folder = REVIEW_ROOT / GUI_EXPERIMENT / GUI_TAG
manifest = json.loads((folder / "review_manifest.json").read_text(encoding="utf-8"))
print(json.dumps(manifest["summary"], indent=1, ensure_ascii=False))

with np.load(folder / "review_data.npz") as z:
    n3, xyz, h, t = z["nodes3d"], z["xyz"], int(z["head_node"]), int(z["tail_node"])
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    length = np.linalg.norm(n3[:, :, h] - n3[:, :, t], axis=-1)
    vel = np.full_like(xyz, np.nan)
    vel[5:-5] = (xyz[10:] - xyz[:-10]) / 10
    speed = np.linalg.norm(vel, axis=-1)
    axis = n3[:, :, h] - n3[:, :, t]
    cos = np.einsum("fkc,fkc->fk", axis, vel) / (np.linalg.norm(axis, axis=-1) * speed)
    fast = speed > 0.3
    print(f"\n嘴-尾 3D 长度：中位数 {np.nanmedian(length):.1f} mm（5–95%：{np.nanpercentile(length, 5):.1f}–{np.nanpercentile(length, 95):.1f}）")
    print(f"游动（>0.3 mm/帧）时头朝前的比例：{np.nanmean(cos[fast] > 0):.3f}（{int(np.nansum(fast))} 帧×鱼）")
risks = pd.read_csv(folder / "machine_risk.csv")
print(f"\n风险点 {len(risks)} 个，共 {int((risks.frame_end - risks.frame_start + 1).sum())} 帧")
risks.head(20)

{
 "orientation_frames": 14935,
 "frames_with_any_view_auto_flipped": 92,
 "orientation_override_frames": 0,
 "risk_windows": 6,
 "risk_window_frames": 1897,
 "risk_types": {
  "close": 2,
  "missing_view": 5
 },
 "seconds": 3.5
}

嘴-尾 3D 长度：中位数 25.3 mm（5–95%：23.1–26.5）
游动（>0.3 mm/帧）时头朝前的比例：0.988（3056 帧×鱼）

风险点 6 个，共 1897 帧


,risk_id,frame_start,frame_end,frame_peak,types,fish,score,details
0,R00001,171970,172091,172020,missing_view,ID_2,22.000000,seen in fewer than 3 views for 22 frames
1,R00002,172270,172869,172320,missing_view,ID_1;ID_2;ID_3,22.000000,seen in fewer than 3 views for 22 frames | see...
2,R00003,173533,173688,173583,missing_view,ID_1,56.000000,seen in fewer than 3 views for 56 frames
3,R00004,173914,174482,174194,close;missing_view,ID_1|ID_2;ID_2,14.558181,seen in fewer than 3 views for 179 frames | se...
4,R00005,174523,174849,174622,close,ID_1|ID_2,14.274614,min 3D distance 14.3 mm | min 3D distance 18.4...
5,R00006,174888,175010,174938,missing_view,ID_2,23.000000,seen in fewer than 3 views for 23 frames


## 步骤 7 — 复制审阅包到移动硬盘 / 把修正记录拷回 Z:

- `TO_SSD`：把 `<实验>/proxy/` 和 `<实验>/<标签>/` 复制到 `SSD_ROOT`，保持同样的文件夹结构（已存在且大小相同的视频跳过）
- `FROM_SSD`：只把修正记录（`human_events.csv`、`coverage.csv`、`session_log.csv`）从移动硬盘拷回 `REVIEW_ROOT`。Z: 上的记录如果比移动硬盘上的新，就不覆盖并提示
- 每个实验约 3 GB（代理视频）

In [7]:
import shutil

from scripts.review.ledger import LEDGER_FILES

COPY_EXPERIMENTS, COPY_TAGS = [GUI_EXPERIMENT], [GUI_TAG]
TO_SSD, FROM_SSD = False, False


def copy_tree(src, dst):
    n = 0
    for f in src.rglob("*"):
        if f.is_dir() or f.suffix == ".lock" or ".partial" in f.name:
            continue
        target = dst / f.relative_to(src)
        if target.exists() and target.stat().st_size == f.stat().st_size and f.name not in LEDGER_FILES:
            continue
        if f.name in LEDGER_FILES and target.exists() and target.stat().st_mtime > f.stat().st_mtime:
            print(f"⚠️ 移动硬盘上的 {target} 更新，没有覆盖（先用 FROM_SSD 拷回）")
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, target)
        n += 1
    return n


if SSD_ROOT is None:
    print("未设置 SSD_ROOT：在步骤 1 填移动硬盘上的文件夹，例如 'E:/medaka_review'")
else:
    ssd = Path(SSD_ROOT)
    for exp in COPY_EXPERIMENTS:
        if TO_SSD:
            for part in ["proxy", *COPY_TAGS]:
                if (REVIEW_ROOT / exp / part).exists():
                    print(f"{exp}/{part}: 复制 {copy_tree(REVIEW_ROOT / exp / part, ssd / exp / part)} 个文件 → {ssd / exp / part}")
        if FROM_SSD:
            for tag in COPY_TAGS:
                for name in LEDGER_FILES:
                    src, dst = ssd / exp / tag / name, REVIEW_ROOT / exp / tag / name
                    if not src.exists():
                        continue
                    if dst.exists() and dst.stat().st_mtime > src.stat().st_mtime:
                        print(f"⚠️ {dst} 比移动硬盘上的新，没有覆盖（是不是在别的电脑上也审过？）")
                        continue
                    shutil.copy2(src, dst)
                    print(f"拷回 {src} → {dst}")
    if not (TO_SSD or FROM_SSD):
        print("把 TO_SSD 或 FROM_SSD 改成 True 再运行")

未设置 SSD_ROOT：在步骤 1 填移动硬盘上的文件夹，例如 'E:/medaka_review'


## 步骤 8 — 在本机打开审阅界面

- 打开 `REVIEW_ROOT/<GUI_EXPERIMENT>/<GUI_TAG>`（默认是 20 秒小测试，用来试界面）
- 界面里按 **F1** 看全部按键；第一次打开：点一条鱼，按 `1`（F）/ `2`（M1）/ `3`（M2），三条鱼都指定后按空格播放
- 修正记录直接写在这个文件夹里；笔记本上的用法见最后一节

In [8]:
if not OPEN_GUI:
    print("未打开：把步骤 1 的 OPEN_GUI 改成 True 再运行本格")
else:
    gui_folder = REVIEW_ROOT / GUI_EXPERIMENT / GUI_TAG
    subprocess.Popen([sys.executable, "-m", "scripts.review.review_gui", str(gui_folder)], cwd=REPO)
    print("已打开：", gui_folder)

未打开：把步骤 1 的 OPEN_GUI 改成 True 再运行本格


## 步骤 9 — 应用修正

- 输入：`review_data.npz` + `human_events.csv` + `coverage.csv`（在移动硬盘上审的，先用步骤 7 的 `FROM_SSD` 拷回）
- 输出：`<实验>/<标签>/<标签>_3d_reviewed.csv`，每帧每条轨迹一行：`label`（F / M1 / M2 / `-` 不是鱼 / 空 = 还没指定）、中心点 `X/Y/Z`、各节点 `mouth_X … tail_Z`（已套用人工头尾翻转）、`uncertain`、`bad3d`、`watched`（≤1 倍速看过）

In [9]:
from scripts.review.ledger import apply_events

APPLY_EXPERIMENT, APPLY_TAG = GUI_EXPERIMENT, GUI_TAG

folder = REVIEW_ROOT / APPLY_EXPERIMENT / APPLY_TAG
with np.load(folder / "review_data.npz") as z:
    data = {k: z[k] for k in z.files}
events = read_csv(folder / "human_events.csv", EVENT_COLUMNS)
coverage = read_csv(folder / "coverage.csv", COVERAGE_COLUMNS)
reviewed = apply_events(data, events, coverage)
out_csv = folder / f"{APPLY_TAG}_3d_reviewed.csv"
reviewed.to_csv(out_csv, index=False)
print(f"{len(events)} 条修正记录 → {out_csv}（{len(reviewed)} 行）")
print("各身份的 帧×轨迹 数：", reviewed["label"].replace("", "未指定").value_counts().to_dict())

0 条修正记录 → \\bucket.oist.jp\bucket\StephensU\medaka\review_packages\20260112_P_1_1_FCMC\s14_bottomup_3node_reviewed_test\s14_bottomup_3node_reviewed_test_3d_reviewed.csv（14940 行）
各身份的 帧×轨迹 数： {'未指定': 14940}


## 步骤 10 — 机器风险点 vs 人工修正

- 第一次给每条轨迹指定身份不算“修正”；之后的每次身份对调、头尾翻转、标不确定、标 3D 不可信才算
- `corrections_inside_risk` / `corrections_outside_risk`：修正落在机器风险点里 / 外的个数；`risk_recall` = 机器风险点抓住的修正比例
- `risk_windows_with_correction`：有多少个风险点真的需要修正（其余是误报）
- `watched_fraction`：≤1 倍速看过的比例（相对有追踪数据的范围）

In [10]:
from scripts.review.ledger import compare_with_machine

risks = pd.read_csv(folder / "machine_risk.csv")
summary = compare_with_machine(events, risks, coverage, int(data["frames"][0]), int(data["frames"][-1]))
print(json.dumps(summary, indent=1, ensure_ascii=False))

{
 "frames": 4980,
 "watched_fraction": 0.0,
 "risk_windows": 6,
 "risk_windows_with_correction": 0,
 "corrections": 0,
 "corrections_by_kind": {},
 "corrections_inside_risk": 0,
 "corrections_outside_risk": 0,
 "risk_recall": null
}


## 步骤 11 — 停止后台任务

把 `STOP_TASK` 改成 `"review_builder"` 再运行（先运行步骤 1–4）。正在转码的那个视角作废，下次启动从头重做。

In [11]:
STOP_TASK = None   # "review_builder"

if STOP_TASK is None:
    print("未停止：把 STOP_TASK 改成要停的任务名再运行本格")
else:
    stop_background(STOP_TASK)

未停止：把 STOP_TASK 改成要停的任务名再运行本格


## 笔记本（或其他电脑）上审阅

**第一次（每台电脑一次）**
1. 装 Python 3.11（python.org 的 Windows 安装包，勾选 “Add python.exe to PATH”）
2. 把仓库 clone（或整个复制）到笔记本
3. 在仓库根目录打开 PowerShell：
   ```powershell
   py -3.11 -m venv .venv-review
   .venv-review\Scripts\python.exe -m pip install -r environment\requirements-review.txt
   ```
   只装 numpy / pandas / opencv / PyQt5，几分钟，不需要显卡、torch、SLEAP

**每次审阅**
1. 在台式机上用步骤 7（`TO_SSD`）把审阅包复制到移动硬盘
2. 笔记本插上移动硬盘，在仓库根目录：
   ```powershell
   .venv-review\Scripts\python.exe -m scripts.review.review_gui E:\medaka_review\20260112_P_1_1_FCMC\s14_bottomup_3node_reviewed_test
   ```
   （盘符按实际情况改；不写路径会弹出选择文件夹的对话框）
3. 审完关掉窗口（自动保存）；回到台式机用步骤 7（`FROM_SSD`）把修正记录拷回 Z:

**按键**（界面里按 F1 也能看）

| 按键 | 作用 |
|---|---|
| 空格 | 播放 / 暂停（1 倍速） |
| L / J / K | 正放 / 倒放（再按一次加速）/ 暂停 |
| ↑ / ↓ | 加速 / 减速（超过 1 倍速的部分不算“看过”） |
| → / ← | 下一个 / 上一个显示帧（约 30 fps 的一步 = 6 个原始帧） |
| . / , | 下一个 / 上一个原始帧（逐帧精确检查） |
| Shift + → / ← | 前进 / 后退 1 秒 |
| 鼠标点击 / Tab / Esc | 选中鱼（任一视角都可以点）/ 轮流选中 / 取消选中 |
| 1 / 2 / 3 | 选中的鱼从这一帧起是 F / M1 / M2（原来持有这个身份的鱼自动对调） |
| 0 | 选中的轨迹从这一帧起不是鱼（例如放鱼时的手或网） |
| H | 选中的鱼从这一帧起头尾对调（之后再按一次取消） |
| U | 选中的鱼标“不确定”（不选 = 三条鱼都标），再按一次结束 |
| X | 选中的鱼标“3D 不可信”，再按一次结束 |
| N / P | 下一个 / 上一个机器风险点 |
| W | 跳到下一个还没看过的位置 |
| A | 风险点里自动减速到 0.5 倍（默认开） |
| O | 显示 / 隐藏标注 |
| G | 跳到指定帧 |
| Ctrl + Z | 撤销这次审阅里的最后一个修正 |

**身份对调发生在两次显示之间时**：按键时程序自动在“上一个显示帧 → 当前帧”之间找两条鱼 3D 距离最近的那一帧作为对调的准确帧，两个帧号都会记录。